# MAIN

## Imports

In [1]:
#imports

import pandas as pd
import numpy as np 

#for importing the preprocessor done in the preprocessing folder
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessorLOEE_OH,preprocessorClassic,preprocessorLOEEALL,preprocessorLOEE_oh_TE_te, preprocessorLOEE_te_TE_oh,preprocessorTEALL, orig_onehot_enc_cols, orig_target_enc_cols, original_preprocessorLOEEALL, onehot_enc_cols, target_enc_cols






from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, make_scorer, cohen_kappa_score


## Dataset splitting

In [ ]:
# DON'T EXECUTE AGAIN, THE SPLIT IS ALREADY DONE AND CAN BE LOADED USING ITS SPECIFIC FILES
'''
df= pd.read_parquet("../data/cleaned/cleaned_df_final.parquet")
#print(df.columns.tolist())

#1. Stratified split 
X = df.drop("AdoptionSpeed", axis=1)
y = df["AdoptionSpeed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

#We store the partiions of the final df
X_train.to_parquet("../data/cleaned/X_train.parquet", index=False)
X_test.to_parquet("../data/cleaned/X_test.parquet", index=False)
y_train.to_frame().to_parquet("../data/cleaned/y_train.parquet", index=False)
y_test.to_frame().to_parquet("../data/cleaned/y_test.parquet", index=False)
'''

(10045, 32)
(10045,)
(4948, 32)
(4948,)


## Dataset loading

In [3]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)



(10045, 32)
(10045,)
(4948, 32)
(4948,)


## Baselines models with training data

scoring for all the models

In [4]:
qwk = make_scorer(cohen_kappa_score, weights="quadratic")
used_scores = {
            "f1_macro": "f1_macro",
            "QWK": qwk
        }



### RandomForest Baseline

In [5]:



#2. Baseline model (RandomForestClassifier with default hyperparameters)
baseline_clf = RandomForestClassifier(random_state=42)

pipe0 = Pipeline (steps=[("preprocessorLOEEALL",preprocessorClassic), ("rf",baseline_clf)])
pipe1 = Pipeline (steps=[("preprocessorLOEEALL",preprocessorLOEEALL), ("rf",baseline_clf)])
pipe2 = Pipeline (steps=[("preprocessorLOEEALL",preprocessorLOEE_OH), ("rf",baseline_clf)])
pipe3 = Pipeline (steps=[("preprocessorLOEE_oh_TE_te",preprocessorLOEE_oh_TE_te), ("rf",baseline_clf)])
pipe4 = Pipeline (steps=[("preprocessorLOEE_te_TE_oh",preprocessorLOEE_te_TE_oh), ("rf",baseline_clf)])
pipe5 = Pipeline (steps=[("preprocessorTEALL",preprocessorTEALL), ("rf",baseline_clf)])

pipelines = [pipe0,pipe1,pipe2,pipe3,pipe4,pipe5]


skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

i = 0
for pipe in pipelines: 
    scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=used_scores, return_train_score=True) 
    print(f"\nPipe{i}...............")
    print(f"TRAIN: F1-macro: {round(np.mean(scores['train_f1_macro']),4)}, QWK mean: {round(np.mean(scores['train_QWK']),4)}, QWK std: {round(np.std(scores['train_QWK']),4)}")
    print(f"TEST: F1-macro: {round(np.mean(scores['test_f1_macro']),4)}, QWK mean: {round(np.mean(scores['test_QWK']),4)}, QWK std: {round(np.std(scores['test_QWK']),4)}")
    i = i+1

"""

Pipe0...............
TRAIN: F1-macro: 1.0, QWK mean: 1.0, QWK std: 0.0
TEST: F1-macro: 0.3256, QWK mean: 0.3919, QWK std: 0.0149

Pipe1............... THE PREPROCESSOR ONLY USING LEAVE ONE OUT ENCODING FOR ALL CATEGORICAL COLUMNS ACHIEVES THE BEST RESULTS AVOIDING HUGE OVERFITTING AND PROVIDING A GOOD QWK SCORE FOR THE BASELINE MODEL
.................... SO WE WILL BE USING THIS PREPROCESSOR FOR THE REST OF THE PROJECT
TRAIN: F1-macro: 0.4709, QWK mean: 0.5879, QWK std: 0.0046
TEST: F1-macro: 0.3114, QWK mean: 0.3883, QWK std: 0.015

Pipe2...............
TRAIN: F1-macro: 0.9238, QWK mean: 0.9269, QWK std: 0.0033
TEST: F1-macro: 0.3217, QWK mean: 0.3913, QWK std: 0.0159

Pipe3...............
TRAIN: F1-macro: 0.6853, QWK mean: 0.8278, QWK std: 0.0048
TEST: F1-macro: 0.2944, QWK mean: 0.3758, QWK std: 0.0076

Pipe4...............
TRAIN: F1-macro: 0.8851, QWK mean: 0.8914, QWK std: 0.0056
TEST: F1-macro: 0.3246, QWK mean: 0.3862, QWK std: 0.0138

Pipe5...............
TRAIN: F1-macro: 1.0, QWK mean: 1.0, QWK std: 0.0
TEST: F1-macro: 0.321, QWK mean: 0.3878, QWK std: 0.0092


"""


Pipe0...............
TRAIN: F1-macro: 1.0, QWK mean: 1.0, QWK std: 0.0
TEST: F1-macro: 0.3256, QWK mean: 0.3919, QWK std: 0.0149

Pipe1...............
TRAIN: F1-macro: 0.4709, QWK mean: 0.5879, QWK std: 0.0046
TEST: F1-macro: 0.3114, QWK mean: 0.3883, QWK std: 0.015

Pipe2...............
TRAIN: F1-macro: 0.9238, QWK mean: 0.9269, QWK std: 0.0033
TEST: F1-macro: 0.3217, QWK mean: 0.3913, QWK std: 0.0159

Pipe3...............
TRAIN: F1-macro: 0.6853, QWK mean: 0.8278, QWK std: 0.0048
TEST: F1-macro: 0.2944, QWK mean: 0.3758, QWK std: 0.0076

Pipe4...............
TRAIN: F1-macro: 0.8851, QWK mean: 0.8914, QWK std: 0.0056
TEST: F1-macro: 0.3246, QWK mean: 0.3862, QWK std: 0.0138

Pipe5...............
TRAIN: F1-macro: 1.0, QWK mean: 1.0, QWK std: 0.0
TEST: F1-macro: 0.321, QWK mean: 0.3878, QWK std: 0.0092


'\n\nPipe0...............\nTRAIN: F1-macro: 1.0, QWK mean: 1.0, QWK std: 0.0\nTEST: F1-macro: 0.3256, QWK mean: 0.3919, QWK std: 0.0149\n\nPipe1............... THE PREPROCESSOR ONLY USING LEAVE ONE OUT ENCODING FOR ALL CATEGORICAL COLUMNS ACHIEVES THE BEST RESULTS AVOIDING HUGE OVERFITTING AND PROVIDING A GOOD QWK SCORE FOR THE BASELINE MODEL\n.................... SO WE WILL BE USING THIS PREPROCESSOR FOR THE REST OF THE PROJECT\nTRAIN: F1-macro: 0.4709, QWK mean: 0.5879, QWK std: 0.0046\nTEST: F1-macro: 0.3114, QWK mean: 0.3883, QWK std: 0.015\n\nPipe2...............\nTRAIN: F1-macro: 0.9238, QWK mean: 0.9269, QWK std: 0.0033\nTEST: F1-macro: 0.3217, QWK mean: 0.3913, QWK std: 0.0159\n\nPipe3...............\nTRAIN: F1-macro: 0.6853, QWK mean: 0.8278, QWK std: 0.0048\nTEST: F1-macro: 0.2944, QWK mean: 0.3758, QWK std: 0.0076\n\nPipe4...............\nTRAIN: F1-macro: 0.8851, QWK mean: 0.8914, QWK std: 0.0056\nTEST: F1-macro: 0.3246, QWK mean: 0.3862, QWK std: 0.0138\n\nPipe5............

### Random Baseline (stratified)

In [ ]:
#2. Baseline model (RandomForestClassifier with default hyperparameters)
baseline_clf = DummyClassifier(random_state=42,strategy="stratified")

pipe = Pipeline (steps=[("preprocessor",preprocessorLOEEALL), ("rf",baseline_clf)])

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=used_scores, return_train_score=True) 

print(f"TRAIN: F1-macro: {round(np.mean(scores['train_f1_macro']),4)}, QWK mean: {round(np.mean(scores['train_QWK']),4)}, QWK std: {round(np.std(scores['train_QWK']),4)}")
print(f"TEST: F1-macro: {round(np.mean(scores['test_f1_macro']),4)}, QWK mean: {round(np.mean(scores['test_QWK']),4)}, QWK std: {round(np.std(scores['test_QWK']),4)}")
"""
TRAIN: F1-macro: 0.1991, QWK mean: -0.0052, QWK std: 0.0086
TEST: F1-macro: 0.2018, QWK mean: -0.0049, QWK std: 0.0236
"""

TRAIN: F1-macro: 0.1991, QWK mean: -0.0052, QWK std: 0.0086
TEST: F1-macro: 0.2018, QWK mean: -0.0049, QWK std: 0.0236


'\nResults: F1-macro: 0.2, QWK mean: -0.0, QWK std: 0.02\n'

### Random Baseline (most_frequent)

In [ ]:
#2. Baseline model (RandomForestClassifier with default hyperparameters)
baseline_clf = DummyClassifier(random_state=42, strategy="most_frequent")

pipe = Pipeline (steps=[("preprocessor",preprocessorLOEEALL), ("rf",baseline_clf)])

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_validate(pipe, X_train, y_train, cv=skf, scoring=used_scores, return_train_score=True) 

print(f"TRAIN: F1-macro: {round(np.mean(scores['train_f1_macro']),4)}, QWK mean: {round(np.mean(scores['train_QWK']),4)}, QWK std: {round(np.std(scores['train_QWK']),4)}")
print(f"TEST: F1-macro: {round(np.mean(scores['test_f1_macro']),4)}, QWK mean: {round(np.mean(scores['test_QWK']),4)}, QWK std: {round(np.std(scores['test_QWK']),4)}")
"""
TRAIN: F1-macro: 0.0875, QWK mean: 0.0, QWK std: 0.0
TEST: F1-macro: 0.0875, QWK mean: 0.0, QWK std: 0.0
"""

TRAIN: F1-macro: 0.0875, QWK mean: 0.0, QWK std: 0.0
TEST: F1-macro: 0.0875, QWK mean: 0.0, QWK std: 0.0


'\nResults: F1-macro: 0.09, QWK mean: 0.0, QWK std: 0.0\n'

## RandomForest Baseline with original csv only deleting NA rows without joining with JSONS

In [9]:
original_df = pd.read_csv("../data/data.csv")

#print(original_df.columns.tolist())

In [ ]:
print(f"Shape: {original_df.shape}")
print(f"NA Values per columns {original_df.isna().sum()}")

#we delete rows with NA for this quick baseline model with original data 

original_df = original_df.dropna(axis=0)

print(f"Shape: {original_df.shape}")
print(f"NA Values per columns {original_df.isna().sum()}")


#We also delete the animal names, description, PetId
original_df.drop(columns=["Name"],inplace=True)
original_df.drop(columns=["Description"],inplace=True)
original_df.drop(columns=["PetID"],inplace=True)


#We convert the categorical variable to string so that the preprocessor works properly

all_categorical = orig_onehot_enc_cols + orig_target_enc_cols
original_df[all_categorical] = original_df[all_categorical].astype(str)


print(original_df.columns.tolist())




#Data split
X = original_df.drop("AdoptionSpeed", axis=1)
y = original_df["AdoptionSpeed"]
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)





Shape: (14993, 24)
NA Values per columns Type                0
Name             1265
Age                 0
Breed1              0
Breed2              0
Gender              0
Color1              0
Color2              0
Color3              0
MaturitySize        0
FurLength           0
Vaccinated          0
Dewormed            0
Sterilized          0
Health              0
Quantity            0
Fee                 0
State               0
RescuerID           0
VideoAmt            0
Description        13
PetID               0
PhotoAmt            0
AdoptionSpeed       0
dtype: int64
Shape: (13716, 24)
NA Values per columns Type             0
Name             0
Age              0
Breed1           0
Breed2           0
Gender           0
Color1           0
Color2           0
Color3           0
MaturitySize     0
FurLength        0
Vaccinated       0
Dewormed         0
Sterilized       0
Health           0
Quantity         0
Fee              0
State            0
RescuerID        0
VideoAmt        

In [ ]:

baseline_clf = RandomForestClassifier(random_state=42, class_weight="balanced")

pipe = Pipeline (steps=[("preprocessor",original_preprocessorLOEEALL), ("rf",baseline_clf)])

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
scores = cross_validate(pipe, X_train_orig, y_train_orig, cv=skf, scoring=used_scores, return_train_score=True) 

print(f"TRAIN: F1-macro: {round(np.mean(scores['train_f1_macro']),4)}, QWK mean: {round(np.mean(scores['train_QWK']),4)}, QWK std: {round(np.std(scores['train_QWK']),4)}")
print(f"TEST: F1-macro: {round(np.mean(scores['test_f1_macro']),4)}, QWK mean: {round(np.mean(scores['test_QWK']),4)}, QWK std: {round(np.std(scores['test_QWK']),4)}")

"""
TRAIN: F1-macro: 0.3864, QWK mean: 0.507, QWK std: 0.0163
TEST: F1-macro: 0.3051, QWK mean: 0.3625, QWK std: 0.0124
"""



TRAIN: F1-macro: 0.3864, QWK mean: 0.507, QWK std: 0.0163
TEST: F1-macro: 0.3051, QWK mean: 0.3625, QWK std: 0.0124


'\nRESULTS: F1-macro: 0.32, QWK mean: 0.32, QWK std: 0.01\n'

## Baseline model with the final Test set

In [ ]:
# Baseline model (RandomForestClassifier with default hyperparameters)
baseline_clf = RandomForestClassifier(random_state=42)

pipe = Pipeline (steps=[("preprocessorLOEEALL",preprocessorLOEEALL), ("rf",baseline_clf)])



pipe.fit(X_train, y_train)


# Predictions
train_preds = pipe.predict(X_train)
test_preds = pipe.predict(X_test)

# Train Score
train_f1 = f1_score(y_train, train_preds, average='macro')
train_qwk = cohen_kappa_score(y_train, train_preds, weights='quadratic')

# Final Test Score
test_f1 = f1_score(y_test, test_preds, average='macro')
test_qwk = cohen_kappa_score(y_test, test_preds, weights='quadratic')

print(f"FINAL RESULTS WITH BASELINE RF DEFAULT MODEL:")
print(f"TRAIN -> F1: {round(train_f1, 4)} | QWK: {round(train_qwk, 4)}")
print(f"TEST  -> F1: {round(test_f1, 4)}  | QWK: {round(test_qwk, 4)}")

"""

FINAL RESULTS WITH BASELINE RF DEFAULT MODEL: AS WE CAN SEE THE TRAIN-TEST DIFFERENCE IN QWK MANTAINS NEARLY THE SAME AS IN THE VALIDATION SCORE
TRAIN -> F1: 0.4633 | QWK: 0.5876
TEST  -> F1: 0.3011  | QWK: 0.3778

"""

FINAL RESULTS WITH BASELINE RF DEFAULT MODEL:
TRAIN -> F1: 0.4633 | QWK: 0.5876
TEST  -> F1: 0.3011  | QWK: 0.3778


'\n\n\nPipe1...\nTRAIN: F1-macro: 0.4709, QWK mean: 0.5879, QWK std: 0.0046\nTEST: F1-macro: 0.3114, QWK mean: 0.3883, QWK std: 0.015\n\n'